# 13B — Interactive Debate Agent

这个 Notebook 为13的四个未见动议增加自然语言对话入口和党派辩论报告。用户不需要输入JSON；JSON只用于模型、证据层和LLM之间的内部传递。

第一轮生成完整报告，后续可以继续追问。模拟党派发言会明确标注为生成内容，不冒充真实议员引文。

## 0. 对话方式

这不是只能对话两轮。每发送一条消息都会形成一次新的API调用；Notebook默认保留最近8轮上下文，避免对话过长。可以继续追问，但调用次数和token消耗会随轮数增加。

两种入口：

1. 代码入口：`chat_with_agent(case_id, message)`；
2. 页面入口：运行最后一个Cell，在下拉框选择动议，在文本框输入自然语言并点击发送。

In [ ]:
# 导入报告生成、缓存和Notebook交互需要的库
import hashlib
import json
import os
import re
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 180)

def find_project_dir(start):
    # 从当前目录向上寻找项目根目录
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'processed').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录。')

PROJECT_DIR = find_project_dir(Path.cwd())
SOURCE_DIR = PROJECT_DIR / 'processed' / 'unseen_motion_agent_v1'
OUTPUT_DIR = PROJECT_DIR / 'processed' / 'unseen_motion_debate_agent_v1'
REPORT_DIR = OUTPUT_DIR / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

LLM_MODEL = 'gpt-4o-mini-2024-07-18'
PROMPT_VERSION = '13b-debate-agent-v4-cache-firewall'
ENABLE_LLM = False
RUN_BATCH_REPORTS = False
MAX_NEW_CALLS_PER_SESSION = 24
MAX_HISTORY_TURNS = 8
SPEECH_LENGTH_TARGETS = {
    'high': {'minimum': 500, 'target': '500至750个中文字'},
    'medium': {'minimum': 250, 'target': '250至400个中文字'},
    'low': {'minimum': 100, 'target': '100至200个中文字'},
}
MAX_SPEECH_EXPANSION_RETRIES = 2

print('Project directory:', PROJECT_DIR)
print('Prompt version:', PROMPT_VERSION)
print('LLM enabled:', ENABLE_LLM)
print('Output directory:', OUTPUT_DIR)

## 1. 读取13生成的冻结预测和RAG材料

13B不会重新训练模型，也不会改变预测概率。它只把13已经生成的政策合同、概率和证据组织成更适合用户阅读和追问的回答。

In [ ]:
required_files = {
    'cases': SOURCE_DIR / 'unseen_case_registry_v1.csv',
    'contracts': SOURCE_DIR / 'unseen_case_contracts_v1.csv',
    'predictions': SOURCE_DIR / 'unseen_party_predictions_v1.csv',
    'evidence': SOURCE_DIR / 'unseen_retrieval_evidence_v1.csv',
    'status': SOURCE_DIR / 'unseen_query_evidence_status_v1.csv',
}
missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('请先完整运行13。缺少：' + ' | '.join(missing))

cases = pd.read_csv(required_files['cases'])
contracts = pd.read_csv(required_files['contracts'])
predictions = pd.read_csv(required_files['predictions'])
evidence = pd.read_csv(required_files['evidence'], low_memory=False)
evidence_status = pd.read_csv(required_files['status'])

PARTIES = ['labour', 'conservative', 'liberal-democrat']
eligible_cases = set(contracts.loc[contracts['prediction_eligible'].astype(bool), 'case_id'])
assert set(predictions['case_id']).issubset(eligible_cases)
assert not predictions['challenge_outcome_used'].astype(bool).any()

case_options = {
    f"{row['case_id']} | {row['motion_title']}": row['case_id']
    for _, row in cases.iterrows()
}
display(cases[['case_id', 'motion_date', 'motion_title']])
display(predictions[['case_id', 'party', 'predicted_policy_stance', 'support_probability_policy_object']].round(4))

## 2. 报告结构和安全规则

LLM负责解释和写作，不负责重新预测。立场、概率、Aye/No效果和证据等级都由程序提供。报告会为每个政党生成：模型原因、证据解读、模拟发言、三个debate points、可能的反驳和回应。

In [ ]:
PARTY_SECTION_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'model_reasoning': {'type': 'string'},
        'evidence_interpretation': {'type': 'string'},
        'simulated_spokesperson_statement': {'type': 'string'},
        'debate_points': {
            'type': 'array', 'items': {'type': 'string'},
            'minItems': 3, 'maxItems': 3,
        },
        'likely_counterargument': {'type': 'string'},
        'rebuttal': {'type': 'string'},
        'citation_ids': {'type': 'array', 'items': {'type': 'string'}},
        'limitations': {'type': 'string'},
    },
    'required': [
        'model_reasoning', 'evidence_interpretation',
        'simulated_spokesperson_statement', 'debate_points',
        'likely_counterargument', 'rebuttal',
        'citation_ids', 'limitations',
    ],
}

REPORT_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'motion_interpretation': {'type': 'string'},
        'policy_effect_explanation': {'type': 'string'},
        'overall_evidence_assessment': {'type': 'string'},
        'labour': PARTY_SECTION_SCHEMA,
        'conservative': PARTY_SECTION_SCHEMA,
        'liberal_democrat': PARTY_SECTION_SCHEMA,
        'cross_party_comparison': {'type': 'string'},
        'uncertainty_and_limits': {'type': 'string'},
    },
    'required': [
        'motion_interpretation', 'policy_effect_explanation',
        'overall_evidence_assessment', 'labour', 'conservative',
        'liberal_democrat', 'cross_party_comparison',
        'uncertainty_and_limits',
    ],
}

FOLLOW_UP_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'answer': {'type': 'string'},
        'citation_ids': {'type': 'array', 'items': {'type': 'string'}},
        'uncertainty': {'type': 'string'},
    },
    'required': ['answer', 'citation_ids', 'uncertainty'],
}

SPEECH_EXPANSION_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'opening_position': {'type': 'string'},
        'policy_effect_argument': {'type': 'string'},
        'substantive_policy_case': {'type': 'string'},
        'response_to_opponents': {'type': 'string'},
        'closing_appeal': {'type': 'string'},
    },
    'required': [
        'opening_position', 'policy_effect_argument',
        'substantive_policy_case', 'response_to_opponents',
        'closing_appeal',
    ],
}

PARTY_FIELD = {
    'labour': 'labour',
    'conservative': 'conservative',
    'liberal-democrat': 'liberal_democrat',
}
PARTY_LABEL = {
    'labour': 'Labour',
    'conservative': 'Conservative',
    'liberal-democrat': 'Liberal Democrat',
}

In [ ]:
def clean_value(value):
    # 把pandas缺失值转换为JSON可接受的None
    return None if pd.isna(value) else value

def build_speech_length_plan(user_request):
    # 用户点名的政党重点展开；没有点名时，核心执政与主要反对党使用中等篇幅
    text = str(user_request).lower()
    aliases = {
        'labour': ['labour', '工党'],
        'conservative': ['conservative', 'tory', '保守党'],
        'liberal-democrat': ['liberal democrat', 'libdem', '自由民主党'],
    }
    all_detailed = any(word in text for word in ['三个党都详细', '三党都详细', '详细分析三个', '深入分析三个', '长篇'])
    explicitly_named = {
        party for party, names in aliases.items()
        if any(name in text for name in names)
    }
    plan = {}
    for party in PARTIES:
        if all_detailed or party in explicitly_named:
            priority = 'high'
        elif explicitly_named:
            priority = 'low'
        elif party in {'labour', 'conservative'}:
            priority = 'medium'
        else:
            priority = 'low'
        plan[party] = {'priority': priority, **SPEECH_LENGTH_TARGETS[priority]}
    return plan

def case_packet(case_id, user_request):
    # 组装单个案例的冻结预测、证据和自然语言请求
    case = cases.set_index('case_id').loc[case_id].to_dict()
    contract = contracts.set_index('case_id').loc[case_id].to_dict()
    pred = predictions.loc[predictions['case_id'].eq(case_id)].copy()
    ev = evidence.loc[evidence['case_id'].eq(case_id)].copy()
    status = evidence_status.loc[evidence_status['case_id'].eq(case_id)].copy()

    prediction_rows = []
    for _, row in pred.iterrows():
        prediction_rows.append({
            'party': row['party'],
            'party_role': row['party_role'],
            'predicted_stance': row['predicted_policy_stance'],
            'support_probability': round(float(row['support_probability_policy_object']), 4),
            'role_model_probability': round(float(row['role_probability']), 4),
            'recent_100_prior_probability': round(float(row['recent_prior_probability']), 4),
            'raw_vote_prediction': row['raw_vote_prediction'],
            'aye_probability': round(float(row['aye_probability']), 4),
        })

    evidence_rows = []
    for _, row in ev.iterrows():
        evidence_rows.append({
            'evidence_id': f"{row['query_party']}:{row['evidence_id']}",
            'party': row['query_party'],
            'source_type': row['source_type'],
            'evidence_role': row['evidence_role'],
            'title': clean_value(row.get('title')),
            'date': str(pd.Timestamp(row['source_date']).date()),
            'stance_label': clean_value(row.get('stance_label')),
            'excerpt': str(row['evidence_text'])[:900],
        })

    aye_effect = int(contract['aye_effect_on_object'])
    vote_effect = {
        'aye': '支持政策对象' if aye_effect == 1 else '反对政策对象',
        'no': '反对政策对象' if aye_effect == 1 else '支持政策对象',
        'aye_effect_on_object': aye_effect,
    }
    return {
        'user_request': user_request,
        'motion': {
            'case_id': case_id,
            'date': case['motion_date'],
            'title': case['motion_title'],
            'text': case['motion_text'],
        },
        'validated_contract': {key: clean_value(value) for key, value in contract.items()},
        'vote_effect': vote_effect,
        'speech_length_plan': build_speech_length_plan(user_request),
        'frozen_predictions': prediction_rows,
        'evidence_status': status.to_dict('records'),
        'retrieved_material': evidence_rows,
    }

def allowed_citations(packet, party=None):
    ids = {row['evidence_id'] for row in packet['retrieved_material']}
    if party is not None:
        prefix = party + ':'
        ids = {item for item in ids if item.startswith(prefix)}
    return ids

def sanitize_report(payload, packet):
    # 无效引用会被删除，而不是让整批任务因为一个格式问题停止
    result = dict(payload)
    for party in PARTIES:
        field = PARTY_FIELD[party]
        section = dict(result[field])
        valid = allowed_citations(packet, party)
        section['citation_ids'] = [item for item in section['citation_ids'] if item in valid]
        result[field] = section
    return result

def get_client():
    if not ENABLE_LLM:
        raise RuntimeError('请先把ENABLE_LLM改成True并重新运行配置Cell。')
    key = os.environ.get('OPENAI_API_KEY', '').strip()
    if not key:
        key = getpass('请粘贴OpenAI API key：').strip()
        if not key:
            raise EnvironmentError('没有提供API key。')
        os.environ['OPENAI_API_KEY'] = key
    from openai import OpenAI
    return OpenAI(api_key=key)

REPORT_CACHE_PATH = OUTPUT_DIR / 'debate_agent_reports_v1.jsonl'
FOLLOW_UP_CACHE_PATH = OUTPUT_DIR / 'debate_agent_followups_v1.jsonl'
new_calls_this_session = 0

def load_jsonl_cache(path):
    cache = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                record = json.loads(line)
                cache[record['cache_key']] = record
    return cache

report_cache = load_jsonl_cache(REPORT_CACHE_PATH)
follow_up_cache = load_jsonl_cache(FOLLOW_UP_CACHE_PATH)

## 3. 生成完整党派辩论报告

立场和概率直接来自13。LLM只负责解释为什么模型会得到该结果，以及模拟各党如何在辩论中表达和维护该立场。

In [ ]:
REPORT_INSTRUCTIONS = '''
你是英国议会政策分析Agent的报告生成器。

你必须遵守：
1. frozen_predictions中的立场和概率已经冻结，不得修改、重算或覆盖。
2. 分开解释role_model_probability和recent_100_prior_probability如何共同影响最终概率。
3. retrieved_material只是参考材料；相关背景不能写成直接证明。
4. 只能引用提供的evidence_id。找不到可靠证据时要明确说明。
5. simulated_spokesperson_statement是模拟内容，不得写成真实议员原话。
6. 每个政党给出三个可以直接用于辩论的完整debate points；每一点都要解释政策机制或现实影响，不能只写一句口号。
7. 严格按照case_packet中的speech_length_plan分配篇幅：high写500至750个中文字，medium写250至400字，low写100至200字。重点内容充分展开，非重点内容保持简洁。
8. high发言应有4至6个自然段，medium发言应有2至4段，low发言可以是1至2段。所有发言仍应包含明确立场、政策效果、核心论据和必要的回应。
9. model_reasoning要解释Role XGBoost、最近100次投票先验与50/50融合结果；evidence_interpretation要区分支持材料、挑战材料和普通背景。
10. likely_counterargument应呈现反方最强论证，rebuttal应作出具体回应，不能只说“我们不同意”。
11. 解释Aye和No对政策对象的实际效果，不能把程序性投票方向说反。
12. 不得读取或猜测新案例的真实投票结果。
'''

def safe_refusal_report(case_id):
    contract = contracts.set_index('case_id').loc[case_id]
    case = cases.set_index('case_id').loc[case_id]
    return (
        f"# {case['motion_title']}\n\n"
        f"> 本案例没有生成单一政党概率：{contract['ineligibility_reason']}\n\n"
        '该动议需要先拆成多个独立政策问题，再分别预测。系统仍可展示相关政策材料，'
        '但不会把多个立场压缩成一个可能误导的支持概率。'
    )

def report_cache_key(case_id, user_request):
    raw = f'{PROMPT_VERSION}|{case_id}|{user_request}'
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def generate_debate_report(case_id, user_request=None):
    global new_calls_this_session
    if user_request is None:
        user_request = '分析这项动议，各党可能持什么立场？解释政策效果，并模拟各党的议会发言、debate points和辩护。'
    if case_id not in set(cases['case_id']):
        raise KeyError(f'未知case_id：{case_id}')
    if case_id not in eligible_cases:
        return {'case_id': case_id, 'markdown': safe_refusal_report(case_id), 'payload': None, 'cached': True}

    packet = case_packet(case_id, user_request)
    cache_key = report_cache_key(case_id, user_request)
    if cache_key in report_cache:
        record = report_cache[cache_key]
        cached_payload = record.get('payload')
        cached_lengths = speech_lengths(cached_payload) if cached_payload else {}
        cache_is_long_enough = bool(cached_payload) and all(
            cached_lengths.get(party, 0) >= int(packet['speech_length_plan'][party]['minimum'])
            for party in PARTIES
        )
        if cache_is_long_enough:
            return {'case_id': case_id, 'markdown': record['markdown'], 'payload': cached_payload, 'cached': True}
        # 旧缓存若未达到当前篇幅标准，只在内存中作废；原始记录仍保留用于审计
        report_cache.pop(cache_key, None)
        print(f'{case_id}的缓存发言过短，正在按当前标准重新生成。')
    if new_calls_this_session >= MAX_NEW_CALLS_PER_SESSION:
        raise RuntimeError('已达到本次Notebook的新调用上限。')

    client = get_client()
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=REPORT_INSTRUCTIONS,
        input=json.dumps(packet, ensure_ascii=False, indent=2),
        text={'format': {'type': 'json_schema', 'name': 'debate_report', 'strict': True, 'schema': REPORT_SCHEMA}},
        temperature=0,
        max_output_tokens=9000,
        store=False,
    )
    if getattr(response, 'status', None) == 'incomplete':
        raise RuntimeError('LLM输出未完成。结果没有写入缓存，请减少问题范围后重试。')
    payload = sanitize_report(json.loads(response.output_text), packet)
    calls_used = 1
    total_input_tokens = int(getattr(response.usage, 'input_tokens', 0) or 0)
    total_output_tokens = int(getattr(response.usage, 'output_tokens', 0) or 0)

    # 只扩写过短的政党发言，避免为了一个短段落重写整份报告
    length_plan = packet['speech_length_plan']
    for party in PARTIES:
        field = PARTY_FIELD[party]
        for _ in range(MAX_SPEECH_EXPANSION_RETRIES):
            current_length = speech_lengths(payload)[party]
            if current_length >= int(length_plan[party]['minimum']):
                break
            if new_calls_this_session + calls_used >= MAX_NEW_CALLS_PER_SESSION:
                break
            party_packet = {
                'motion': packet['motion'],
                'validated_contract': packet['validated_contract'],
                'vote_effect': packet['vote_effect'],
                'party': party,
                'frozen_prediction': next(row for row in packet['frozen_predictions'] if row['party'] == party),
                'party_evidence': [row for row in packet['retrieved_material'] if row['party'] == party],
                'current_party_analysis': payload[field],
                'required_target': length_plan[party],
            }
            expanded = client.responses.create(
                model=LLM_MODEL,
                instructions=(
                    '你只负责扩写一个政党的模拟议会发言。不得改变冻结立场、概率、投票效果或证据等级。'
                    '输出五个能够连续组成一篇发言的段落：立场开场、政策效果、实质政策论证、回应反方、结尾主张。'
                    '严格达到required_target指定的篇幅；不得把模拟文字说成真实引文，不得引入输入之外的事实。'
                ),
                input=json.dumps(party_packet, ensure_ascii=False, indent=2),
                text={'format': {'type': 'json_schema', 'name': 'expanded_party_speech', 'strict': True, 'schema': SPEECH_EXPANSION_SCHEMA}},
                temperature=0,
                max_output_tokens=2600,
                store=False,
            )
            calls_used += 1
            total_input_tokens += int(getattr(expanded.usage, 'input_tokens', 0) or 0)
            total_output_tokens += int(getattr(expanded.usage, 'output_tokens', 0) or 0)
            if getattr(expanded, 'status', None) == 'incomplete':
                continue
            speech_parts = json.loads(expanded.output_text)
            payload[field]['simulated_spokesperson_statement'] = '\n\n'.join(speech_parts.values())

        # 两次专门扩写后仍偏短时，用同一份已生成论证补成完整发言，不再额外收费
        if speech_lengths(payload)[party] < int(length_plan[party]['minimum']):
            payload[field]['simulated_spokesperson_statement'] = assemble_speech_from_section(payload[field])
    markdown = render_report(case_id, payload)
    record = {
        'cache_key': cache_key, 'case_id': case_id, 'prompt_version': PROMPT_VERSION,
        'user_request': user_request, 'model': LLM_MODEL, 'payload': payload,
        'markdown': markdown,
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens,
        'api_calls_used': calls_used,
        'speech_lengths': speech_lengths(payload),
        'speech_length_plan': packet['speech_length_plan'],
        'speech_length_gate_passed': all(
            speech_lengths(payload)[party] >= int(packet['speech_length_plan'][party]['minimum'])
            for party in PARTIES
        ),
    }
    with REPORT_CACHE_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    report_cache[cache_key] = record
    new_calls_this_session += calls_used
    (REPORT_DIR / f'{case_id}.md').write_text(markdown, encoding='utf-8')
    return {'case_id': case_id, 'markdown': markdown, 'payload': payload, 'cached': False}

def speech_lengths(payload):
    # 按去除空白后的字符数检查发言是否达到展示长度
    return {
        party: len(re.sub(r'\s+', '', str(payload[PARTY_FIELD[party]]['simulated_spokesperson_statement'])))
        for party in PARTIES
    }

def assemble_speech_from_section(section):
    # 复用同一份已生成内容组成完整发言，不添加新事实
    debate_text = '；'.join(section['debate_points'])
    paragraphs = [
        section['simulated_spokesperson_statement'],
        '从政策判断看，' + section['model_reasoning'],
        '我们在本场辩论中的核心理由是：' + debate_text,
        '反对者最可能提出这样的质疑：' + section['likely_counterargument'],
        '我们的回应是：' + section['rebuttal'],
        '就现有材料而言，' + section['evidence_interpretation'],
    ]
    return '\n\n'.join(str(item).strip() for item in paragraphs if str(item).strip())

def render_report(case_id, payload):
    case = cases.set_index('case_id').loc[case_id]
    contract = contracts.set_index('case_id').loc[case_id]
    pred = predictions.loc[predictions['case_id'].eq(case_id)].set_index('party')
    aye_effect = int(contract['aye_effect_on_object'])
    aye_text = '支持政策对象' if aye_effect == 1 else '反对政策对象'
    no_text = '反对政策对象' if aye_effect == 1 else '支持政策对象'
    lines = [
        f"# {case['motion_title']}", '',
        '> 以下党派发言均为AI根据冻结预测和检索材料生成的模拟文本，不是真实议员引文。', '',
        '## 动议和投票效果', '',
        payload['motion_interpretation'], '',
        f"- 政策对象：{contract['policy_object']}",
        f"- 投Aye：{aye_text}",
        f"- 投No：{no_text}", '',
        payload['policy_effect_explanation'], '',
        '## 证据总体情况', '', payload['overall_evidence_assessment'], '',
    ]
    for party in PARTIES:
        row = pred.loc[party]
        section = payload[PARTY_FIELD[party]]
        lines += [
            f"## {PARTY_LABEL[party]}", '',
            f"- 模型预测：{row['predicted_policy_stance']}",
            f"- 支持政策对象的概率：{float(row['support_probability_policy_object']):.1%}",
            f"- Role XGBoost概率：{float(row['role_probability']):.1%}",
            f"- 最近100次投票先验：{float(row['recent_prior_probability']):.1%}",
            f"- 原始投票预测：{row['raw_vote_prediction']}", '',
            '### 为什么模型会这样预测', '', section['model_reasoning'], '',
            '### 证据怎样理解', '', section['evidence_interpretation'], '',
            f"### 模拟党派发言（{speech_lengths(payload)[party]}字）", '', section['simulated_spokesperson_statement'], '',
            '### Debate points', '',
        ]
        lines += [f"{index}. {point}" for index, point in enumerate(section['debate_points'], 1)]
        lines += [
            '', '### 可能面对的反方观点', '', section['likely_counterargument'], '',
            '### 该党可能如何回应', '', section['rebuttal'], '',
            f"证据引用：{', '.join(section['citation_ids']) if section['citation_ids'] else '没有足够的可引用外部材料'}", '',
            f"限制：{section['limitations']}", '',
        ]
    lines += [
        '## 跨党比较', '', payload['cross_party_comparison'], '',
        '## 不确定性和使用限制', '', payload['uncertainty_and_limits'],
    ]
    return '\n'.join(lines)

## 4. 可选：一次生成四个案例的报告

把配置Cell中的 `ENABLE_LLM` 和 `RUN_BATCH_REPORTS` 改成 `True` 后运行。三个可预测案例各调用一次；多议题案例输出安全拒绝说明，不调用LLM。

In [ ]:
batch_rows = []
if RUN_BATCH_REPORTS:
    for case_id in cases['case_id']:
        result = generate_debate_report(case_id)
        batch_rows.append({
            'case_id': case_id,
            'cached': result['cached'],
            'report_path': str(REPORT_DIR / f'{case_id}.md'),
            'markdown': result['markdown'],
        })
        display(Markdown(result['markdown']))
    pd.DataFrame(batch_rows).to_csv(OUTPUT_DIR / 'debate_agent_reports_v1.csv', index=False, encoding='utf-8-sig')
else:
    print('RUN_BATCH_REPORTS=False：没有发生批量LLM调用。')

## 5. 连续自然语言追问

第一次提问会生成完整报告；之后的问题会读取该报告和最近的对话历史。用户仍然只输入自然语言。

In [ ]:
CHAT_STATE = {'case_id': None, 'report': None, 'turns': []}

FOLLOW_UP_INSTRUCTIONS = '''
你是政策分析Agent的追问回答器。只能根据冻结报告、冻结预测和给定证据回答。
不得改变概率、预测立场、Aye/No效果或证据等级。
模拟党派发言必须明确是模拟内容。只能引用允许的evidence_id。
如果问题超出材料范围，要直接说明无法由现有材料确认。
'''

def follow_up_cache_key(case_id, message, turns):
    raw = json.dumps({
        'version': PROMPT_VERSION, 'case_id': case_id,
        'message': message, 'turns': turns[-MAX_HISTORY_TURNS:],
    }, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def answer_follow_up(case_id, message):
    global new_calls_this_session
    packet = case_packet(case_id, message)
    turns = CHAT_STATE['turns'][-MAX_HISTORY_TURNS:]
    cache_key = follow_up_cache_key(case_id, message, turns)
    if cache_key in follow_up_cache:
        return follow_up_cache[cache_key]['markdown']
    if new_calls_this_session >= MAX_NEW_CALLS_PER_SESSION:
        raise RuntimeError('已达到本次Notebook的新调用上限。')

    follow_up_input = {
        'current_question': message,
        'frozen_report': CHAT_STATE['report'],
        'recent_conversation': turns,
        'case_packet': packet,
    }
    client = get_client()
    response = client.responses.create(
        model=LLM_MODEL, instructions=FOLLOW_UP_INSTRUCTIONS,
        input=json.dumps(follow_up_input, ensure_ascii=False, indent=2),
        text={'format': {'type': 'json_schema', 'name': 'debate_follow_up', 'strict': True, 'schema': FOLLOW_UP_SCHEMA}},
        temperature=0, max_output_tokens=1800, store=False,
    )
    payload = json.loads(response.output_text)
    valid = allowed_citations(packet)
    citations = [item for item in payload['citation_ids'] if item in valid]
    markdown = payload['answer']
    if citations:
        markdown += '\n\n证据引用：' + ', '.join(citations)
    if payload['uncertainty']:
        markdown += '\n\n不确定性：' + payload['uncertainty']
    record = {
        'cache_key': cache_key, 'case_id': case_id, 'message': message,
        'payload': payload, 'markdown': markdown,
    }
    with FOLLOW_UP_CACHE_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    follow_up_cache[cache_key] = record
    new_calls_this_session += 1
    return markdown

def chat_with_agent(case_id, message):
    # 切换案例时开启一段新对话；同一案例则继续追问
    if CHAT_STATE['case_id'] != case_id:
        CHAT_STATE['case_id'] = case_id
        CHAT_STATE['report'] = None
        CHAT_STATE['turns'] = []

    if CHAT_STATE['report'] is None:
        result = generate_debate_report(case_id, message)
        answer = result['markdown']
        CHAT_STATE['report'] = answer
    else:
        answer = answer_follow_up(case_id, message)

    CHAT_STATE['turns'].append({'role': 'user', 'content': message})
    CHAT_STATE['turns'].append({'role': 'assistant', 'content': answer})
    CHAT_STATE['turns'] = CHAT_STATE['turns'][-MAX_HISTORY_TURNS:]
    return answer

def reset_chat():
    CHAT_STATE.update({'case_id': None, 'report': None, 'turns': []})
    print('对话上下文已清空。')

## 6. Notebook聊天入口

先在配置Cell把 `ENABLE_LLM=True`。运行下面Cell后，选择案例、输入问题并点击发送。切换案例会自动开启新对话。

In [ ]:
try:
    import ipywidgets as widgets

    case_dropdown = widgets.Dropdown(
        options=list(case_options.items()),
        description='动议：',
        layout=widgets.Layout(width='95%'),
    )
    message_box = widgets.Textarea(
        value='分析这项动议，说明Aye和No的政策效果，并模拟三个政党的发言、debate points和辩护。',
        description='消息：',
        layout=widgets.Layout(width='95%', height='120px'),
    )
    send_button = widgets.Button(description='发送', button_style='primary')
    reset_button = widgets.Button(description='清空对话')
    chat_output = widgets.Output(layout={'border': '1px solid #cccccc', 'padding': '10px'})

    def send_message(_):
        with chat_output:
            message = message_box.value.strip()
            if not message:
                print('请输入问题。')
                return
            print(f'\n你：{message}\n')
            try:
                answer = chat_with_agent(case_dropdown.value, message)
                display(Markdown(answer))
            except Exception as error:
                print(f'调用失败：{type(error).__name__}: {error}')

    def clear_conversation(_):
        reset_chat()
        chat_output.clear_output()

    send_button.on_click(send_message)
    reset_button.on_click(clear_conversation)
    display(widgets.VBox([case_dropdown, message_box, widgets.HBox([send_button, reset_button]), chat_output]))
except ImportError:
    print('当前环境没有ipywidgets。仍可使用：chat_with_agent(case_id, message)')

## 7. 不使用按钮时的调用示例

```python
answer = chat_with_agent(
    'U04_ARMED_FORCES_ILR_FEES',
    '请模拟三个政党的议会发言，并重点解释为什么Labour可能反对。'
)
display(Markdown(answer))

# 在同一案例下继续追问，不需要重新提交JSON
answer = chat_with_agent(
    'U04_ARMED_FORCES_ILR_FEES',
    'Conservative最可能怎样反驳Labour？请用更接近议会辩论的口吻回答。'
)
display(Markdown(answer))
```

切换到另一个 `case_id` 会自动开始新的对话。